# Projeto de Análise Exploratória de Dados (EDA)
## Planejamento — Dataset, Problema e Perguntas Norteadoras

> Este notebook registra a **primeira versão do projeto de EDA** que será desenvolvido no Módulo 4.  
> Aqui não realizamos a análise em si, mas sim o planejamento que vai guiá-la: a escolha do dataset, a definição do problema de negócio e as perguntas que queremos responder com os dados.

---

Um dos erros mais comuns de quem está começando em Ciência de Dados é **partir direto para os gráficos** sem saber o que está procurando.  
A análise exploratória eficiente começa antes do código — ela começa com **perguntas bem formuladas**.

> *"Se eu tivesse uma hora para resolver um problema, passaria 55 minutos pensando nele e 5 minutos pensando em soluções."*  
> — Atribuído a Albert Einstein

---
## 1. O Dataset: Olist Brazilian E-commerce

### 1.1 Origem e contexto

O dataset utilizado neste projeto é o **[Olist Brazilian E-commerce Dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)**, disponibilizado publicamente no Kaggle pela própria empresa.

A **Olist** é uma plataforma de marketplace que conecta pequenos lojistas a grandes canais de venda no Brasil. Em vez de cada lojista criar sua própria loja, eles utilizam a infraestrutura da Olist para vender em múltiplos marketplaces simultaneamente.

Os dados cobrem pedidos realizados **entre setembro de 2016 e outubro de 2018**, com informações sobre:
- status e datas dos pedidos
- localização dos clientes e vendedores
- categorias de produtos
- formas e valores de pagamento
- avaliações e comentários dos clientes

### 1.2 Estrutura do dataset

O dataset é composto por **9 tabelas relacionadas**, conectadas por chaves como `order_id`, `customer_id` e `product_id`. Para o nosso projeto de EDA, trabalharemos principalmente com as seguintes tabelas:

| Arquivo | Descrição | Linhas (aprox.) |
|---|---|---|
| `olist_orders_dataset.csv` | Pedidos, status e datas de entrega | ~100 mil |
| `olist_order_reviews_dataset.csv` | Avaliações e notas dos clientes | ~100 mil |
| `olist_order_payments_dataset.csv` | Formas e valores de pagamento | ~104 mil |
| `olist_customers_dataset.csv` | Localização dos clientes | ~100 mil |
| `olist_order_items_dataset.csv` | Itens e preços de cada pedido | ~113 mil |
| `olist_products_dataset.csv` | Categorias e características dos produtos | ~33 mil |

### 1.3 Por que esse dataset é ideal para uma EDA introdutória?

- **Contexto familiar:** todo mundo já fez uma compra online — o domínio é imediato
- **Dados reais e imperfeitos:** existem valores ausentes, outliers e decisões de limpeza a serem tomadas
- **Riqueza analítica:** permite explorar tempo, espaço, comportamento do consumidor e qualidade de serviço
- **Perguntas de negócio reais:** as análises têm impacto prático — não são exercícios artificiais
- **Escala adequada:** grande o suficiente para ser realista, pequeno o suficiente para rodar em qualquer máquina

---
## 2. Inspeção Inicial do Dataset

Antes de formular o problema com precisão, é fundamental ver o que os dados dizem. Esta seção carrega as tabelas principais e faz uma inspeção rápida para confirmar que temos o que precisamos.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1200)

# Caminho base dos datasets — ajuste se necessário
BASE = Path("../datasets")

print("Bibliotecas carregadas com sucesso.")

In [ ]:
# Carregando as tabelas principais
orders   = pd.read_csv(BASE / "olist_orders_dataset.csv")
reviews  = pd.read_csv(BASE / "olist_order_reviews_dataset.csv")
payments = pd.read_csv(BASE / "olist_order_payments_dataset.csv")
customers = pd.read_csv(BASE / "olist_customers_dataset.csv")
items    = pd.read_csv(BASE / "olist_order_items_dataset.csv")
products = pd.read_csv(BASE / "olist_products_dataset.csv")

print("Tabelas carregadas:")
tabelas = {
    "orders":    orders,
    "reviews":   reviews,
    "payments":  payments,
    "customers": customers,
    "items":     items,
    "products":  products
}
for nome, df in tabelas.items():
    print(f"  {nome:12s}: {df.shape[0]:>7,} linhas × {df.shape[1]:>2} colunas")

In [ ]:
# Visão rápida das tabelas mais importantes para o projeto
print("=== Primeiros registros: orders ===")
display(orders.head(3))

print("\n=== Primeiros registros: reviews ===")
display(reviews.head(3))

print("\n=== Primeiros registros: payments ===")
display(payments.head(3))

In [ ]:
# Verificação rápida de valores ausentes nas tabelas
print("Valores ausentes por tabela:")
for nome, df in tabelas.items():
    total_ausentes = df.isnull().sum().sum()
    if total_ausentes > 0:
        print(f"\n  [{nome}] — {total_ausentes} valores ausentes")
        print(df.isnull().sum()[df.isnull().sum() > 0].to_string())

In [ ]:
# Distribuição de notas de avaliação — primeiro sinal do que vamos explorar
print("Distribuição das notas de avaliação (review_score):")
dist_notas = reviews["review_score"].value_counts().sort_index()
dist_notas_pct = (dist_notas / dist_notas.sum() * 100).round(1)

resumo = pd.DataFrame({"quantidade": dist_notas, "percentual_%": dist_notas_pct})
print(resumo)
print(f"\nNota média geral: {reviews['review_score'].mean():.2f}")

In [ ]:
# Distribuição dos status dos pedidos — entendendo o ciclo de vida
print("Distribuição dos status de pedidos:")
print(orders["order_status"].value_counts())
print(f"\nPercentual de pedidos entregues: {(orders['order_status'] == 'delivered').mean():.1%}")

---
## 3. O Problema de Negócio

### 3.1 Definição do problema

Com base na inspeção inicial dos dados, identificamos que o dataset da Olist oferece uma oportunidade de análise com impacto real:

---

###  Problema central

> **O que determina a satisfação do cliente em um pedido na Olist — e como o desempenho operacional de entrega influencia essa avaliação?**

---

### 3.2 Por que este problema é relevante?

A avaliação do cliente (`review_score`, de 1 a 5 estrelas) é um dos indicadores mais importantes de qualidade em qualquer plataforma de e-commerce. Ela afeta diretamente:

- a **reputação dos vendedores** dentro da plataforma
- a **taxa de recompra** dos clientes
- o **posicionamento dos produtos** nos resultados de busca
- a **percepção geral da Olist** como marketplace confiável

Entender o que faz um cliente dar 1 estrela vs. 5 estrelas é uma pergunta de negócio com valor prático imediato.

### 3.3 Hipóteses iniciais

Antes de analisar os dados, levantamos hipóteses baseadas no senso comum e em conhecimento de domínio:

| # | Hipótese | Raciocínio |
|---|---|---|
| H1 | Pedidos atrasados recebem notas mais baixas | Clientes frustrados com o prazo tendem a avaliar pior |
| H2 | Pedidos mais baratos têm notas mais altas | Expectativas menores são mais fáceis de superar |
| H3 | Categorias de produtos influenciam as notas | Produtos de beleza e moda podem ter mais frustração com tamanho/cor |
| H4 | A nota média varia por região do Brasil | Infraestrutura logística é diferente entre estados |
| H5 | Pedidos com mais itens tendem a ter mais problemas | Mais itens = maior chance de algum ter problema |

A EDA do Módulo 4 vai **confirmar, refutar ou refinar** cada uma dessas hipóteses.

---
## 4. Perguntas Norteadoras

As perguntas norteadoras são o coração do planejamento de uma EDA. Elas transformam o problema amplo em **questões específicas, mensuráveis e respondíveis com os dados disponíveis**.

Organizamos as perguntas em quatro eixos analíticos:

---

###  Eixo 1 — Comportamento dos Pedidos

> *Entender como os pedidos se distribuem no tempo e no espaço.*

**P1.1** — Como evoluiu o volume de pedidos ao longo do tempo? Existe uma tendência de crescimento clara?

**P1.2** — Existem padrões sazonais? Quais meses ou períodos concentram mais pedidos (ex: Black Friday, Natal)?

**P1.3** — Em qual dia da semana e horário do dia os clientes mais compram?

**P1.4** — Quais estados do Brasil concentram mais clientes? A demanda é geograficamente concentrada ou distribuída?

---

###  Eixo 2 — Desempenho Operacional (Entrega)

> *Avaliar a eficiência logística e identificar gargalos.*

**P2.1** — Qual é o prazo médio de entrega? Como ele se distribui?

**P2.2** — Qual percentual de pedidos é entregue antes da data estimada? E após?

**P2.3** — O prazo de entrega varia de acordo com a região de destino? Quais estados têm as entregas mais rápidas e mais lentas?

**P2.4** — O desempenho operacional melhorou ao longo do tempo, à medida que a Olist cresceu?

---

###  Eixo 3 — Satisfação do Cliente (Avaliações)

> *Investigar o que influencia a nota que o cliente dá ao pedido.*

**P3.1** — Qual é a distribuição das notas de avaliação? A maioria dos clientes está satisfeita?

**P3.2** — Pedidos atrasados recebem notas significativamente mais baixas do que pedidos entregues no prazo?

**P3.3** — Existe relação entre o prazo de entrega (em dias) e a nota recebida?

**P3.4** — A nota média varia por estado de destino? Regiões com pior logística têm clientes menos satisfeitos?

**P3.5** — Existe diferença na satisfação entre categorias de produtos?

---

###  Eixo 4 — Perfil Financeiro dos Pedidos

> *Entender o padrão de compra e pagamento dos clientes.*

**P4.1** — Qual é o valor médio dos pedidos? Como ele se distribui?

**P4.2** — Quais são as formas de pagamento mais usadas? O cartão de crédito domina?

**P4.3** — Pedidos pagos em mais parcelas têm alguma relação com a satisfação do cliente?

**P4.4** — Quais categorias de produtos geram maior valor médio por pedido?

---
## 5. Variáveis-Chave do Projeto

Para responder às perguntas acima, precisaremos criar um **DataFrame analítico consolidado**, unindo as principais tabelas do dataset.

### 5.1 Variável-alvo

| Variável | Tabela | Descrição |
|---|---|---|
| `review_score` | `olist_order_reviews_dataset.csv` | Nota de 1 a 5 dada pelo cliente ao pedido |

### 5.2 Variáveis explicativas planejadas

| Variável | Origem | Tipo | Uso esperado |
|---|---|---|---|
| `prazo_entrega_dias` | Calculada (orders) | Numérica | Eixo 2 e 3 |
| `atraso_dias` | Calculada (orders) | Numérica | Eixo 2 e 3 |
| `entregou_no_prazo` | Calculada (orders) | Booleana | Eixo 2 e 3 |
| `order_status` | orders | Categórica | Eixo 1 e 3 |
| `customer_state` | customers | Categórica | Eixo 1 e 3 |
| `payment_type` | payments | Categórica | Eixo 4 |
| `payment_value` | payments | Numérica | Eixo 4 |
| `payment_installments` | payments | Numérica | Eixo 4 |
| `product_category_name` | products | Categórica | Eixo 3 e 4 |
| `price` | items | Numérica | Eixo 4 |
| `ano_compra`, `mes_compra` | Calculadas (orders) | Temporal | Eixo 1 |

### 5.3 Prévia da construção do DataFrame analítico

In [ ]:
# Prévia: construção do DataFrame analítico que será usado na EDA completa
# Esta é a lógica de join que vamos aplicar no Módulo 4

# Passo 1: converter datas no orders
colunas_data = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for col in colunas_data:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

# Passo 2: criar indicadores de entrega
orders["prazo_entrega_dias"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.days

orders["atraso_dias"] = (
    orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]
).dt.days

orders["entregou_no_prazo"] = orders["atraso_dias"] <= 0
orders["ano_compra"]  = orders["order_purchase_timestamp"].dt.year
orders["mes_compra"]  = orders["order_purchase_timestamp"].dt.month

# Passo 3: nota média por pedido (alguns pedidos têm mais de uma avaliação)
nota_por_pedido = (
    reviews
    .groupby("order_id")["review_score"]
    .mean()
    .reset_index()
    .rename(columns={"review_score": "nota_media"})
)

# Passo 4: valor total por pedido (pagamentos)
valor_por_pedido = (
    payments
    .groupby("order_id")
    .agg(
        valor_total=("payment_value", "sum"),
        parcelas_max=("payment_installments", "max"),
        forma_pagamento=("payment_type", lambda x: x.mode()[0])
    )
    .reset_index()
)

# Passo 5: estado do cliente
estado_cliente = customers[["customer_id", "customer_state"]].drop_duplicates()

# Passo 6: unindo tudo
df_eda = (
    orders
    .merge(nota_por_pedido,  on="order_id",   how="left")
    .merge(valor_por_pedido, on="order_id",   how="left")
    .merge(estado_cliente,   on="customer_id", how="left")
)

print(f"DataFrame analítico construído: {df_eda.shape[0]:,} pedidos × {df_eda.shape[1]} colunas")
print()
print("Colunas disponíveis:")
for col in df_eda.columns:
    print(f"  - {col}")

In [ ]:
# Primeiros registros do DataFrame analítico consolidado
df_eda.head(5)

In [ ]:
# Salvando o DataFrame analítico para uso imediato no Módulo 4
caminho_saida = Path("../datasets/olist_eda_consolidado.csv")
caminho_saida.parent.mkdir(parents=True, exist_ok=True)

df_eda.to_csv(caminho_saida, index=False)
print(f" DataFrame analítico salvo em: {caminho_saida}")
print(f"   Linhas:  {df_eda.shape[0]:,}")
print(f"   Colunas: {df_eda.shape[1]}")

---
## 6. Sinais Iniciais nos Dados

Antes de encerrar o planejamento, vamos rodar algumas verificações rápidas para confirmar que nossas hipóteses têm base nos dados. Esses são os **primeiros sinais** — não conclusões, apenas evidências iniciais que motivam a análise completa.

In [ ]:
# Sinal 1: nota média para pedidos no prazo vs. atrasados
# Hipótese H1: pedidos atrasados recebem notas mais baixas

entregues = df_eda[
    (df_eda["order_status"] == "delivered") &
    df_eda["nota_media"].notna() &
    df_eda["entregou_no_prazo"].notna()
]

nota_por_pontualidade = entregues.groupby("entregou_no_prazo")["nota_media"].agg(["mean", "count"])
nota_por_pontualidade.index = ["Atrasado", "No prazo"]
nota_por_pontualidade.columns = ["nota_media", "quantidade"]
nota_por_pontualidade["nota_media"] = nota_por_pontualidade["nota_media"].round(2)

print("Nota média por pontualidade de entrega:")
print(nota_por_pontualidade)
print()
print("→ Sinal: há diferença na nota entre pedidos entregues no prazo e atrasados?")

In [ ]:
# Sinal 2: nota média por forma de pagamento
# Hipótese H4 (financeiro): a forma de pagamento influencia a experiência?

nota_por_pagamento = (
    entregues
    .groupby("forma_pagamento")["nota_media"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "nota_media", "count": "quantidade"})
    .sort_values("nota_media", ascending=False)
    .round(2)
)

print("Nota média por forma de pagamento:")
print(nota_por_pagamento)

In [ ]:
# Sinal 3: prazo médio de entrega por estado
# Hipótese H4 (regional): a logística varia muito por região?

prazo_por_estado = (
    entregues
    .groupby("customer_state")["prazo_entrega_dias"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "prazo_medio_dias", "count": "total_pedidos"})
    .sort_values("prazo_medio_dias", ascending=False)
    .round(1)
)

print("Estados com MAIOR prazo médio de entrega:")
print(prazo_por_estado.head(5))
print()
print("Estados com MENOR prazo médio de entrega:")
print(prazo_por_estado.tail(5))

---
## 7. Mapa do Projeto — O que vem a seguir

Este notebook define o **ponto de partida**. Abaixo está o roteiro que seguiremos no **Módulo 4 — Análise Exploratória de Dados (EDA)**:

```
PLANEJAMENTO (este notebook)
  └─ Dataset definido: Olist Brazilian E-commerce
  └─ Problema formulado: O que determina a satisfação do cliente?
  └─ 16 perguntas norteadoras em 4 eixos
  └─ DataFrame analítico consolidado (df_eda) pronto

MÓDULO 4 — EDA COMPLETA
  ├─ Eixo 1: Análise temporal e geográfica dos pedidos
  ├─ Eixo 2: Desempenho operacional de entrega
  ├─ Eixo 3: Avaliações e satisfação do cliente
  ├─ Eixo 4: Perfil financeiro
  └─ Síntese: insights, limitações e próximos passos
```

### Critérios de sucesso da EDA

Ao final do Módulo 4, esperamos ser capazes de responder:

-  **O prazo de entrega é o principal fator de insatisfação?**
-  **Quais regiões do Brasil têm a pior experiência de entrega?**
-  **Existem categorias de produtos problemáticas?**
-  **Qual é o perfil de um pedido com nota máxima vs. nota mínima?**

> **Nota metodológica:** A EDA não busca provar causalidade — queremos encontrar padrões, levantar hipóteses e comunicar descobertas com clareza. Análises causais mais rigorosas exigiriam técnicas estatísticas e experimentais além do escopo deste curso.

---
## Resumo Executivo

| Item | Definição |
|---|---|
| **Dataset** | Olist Brazilian E-commerce (Kaggle) |
| **Período** | Setembro de 2016 a Outubro de 2018 |
| **Volume** | ~100 mil pedidos, 9 tabelas relacionadas |
| **Problema** | O que determina a satisfação do cliente na Olist? |
| **Variável-alvo** | `review_score` (nota de 1 a 5) |
| **Eixos de análise** | Comportamento, Entrega, Satisfação, Financeiro |
| **Perguntas norteadoras** | 16 perguntas distribuídas em 4 eixos |
| **Hipóteses** | 5 hipóteses a confirmar ou refutar na EDA |
| **Próximo passo** | Módulo 4 — EDA completa com visualizações e storytelling |